# Intro

**Seminar by @Elad_Benjo**

In this notebook I will attempt to develop a new model for nowcsating of the Israeli GDP.

Our efforts would be towards a DFM, since we have various frequencies and high demensional data.

In [2]:
from google.colab import drive
import sys

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/gdp_nowcasting_seminar/src')

Mounted at /content/drive


## Papers

https://www.federalreserve.gov/econres/ifdp/files/ifdp1385.pdf

# Pre-model Prep

In [3]:
import pandas as pd

In [85]:
daily_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/feature_selected/daily_data.pkl')

In [86]:
monthly_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/feature_selected/monthly_data.pkl')

In [87]:
monthly_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 364 entries, 1995-01-01 to 2025-04-01
Freq: MS
Data columns (total 43 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Total refunds from the Income Tax Department                                    339 non-null    float64
 1   self employed returns                                                           339 non-null    float64
 2   Cancellations Deductions                                                        352 non-null    float64
 3   Capital Gas Tax Refunds                                                         351 non-null    float64
 4   Cancellation companies                                                          352 non-null    float64
 5   Purchase returns                                                                351 non-null    flo

0, 1, 2, 3, 4, 5, 6, 7, 10, 16, 19, 20, 21, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42

In [88]:
monthly_df.iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 10, 16, 19, 20, 21, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]] = monthly_df.iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 10, 16, 19, 20, 21, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]].shift(1)

In [50]:
import preprocessing

In [89]:
df_X = preprocessing.merge_series_freq({'d': daily_df, 'm': monthly_df})

In [90]:
df_X.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9482 entries, 1995-01-01 to 2025-08-01
Data columns (total 47 columns):
 #   Column                                                                            Non-Null Count  Dtype  
---  ------                                                                            --------------  -----  
 0   d_TA125                                                                           4879 non-null   float64
 1   d_SP500                                                                           6135 non-null   float64
 2   d_USD_ILS                                                                         5969 non-null   float64
 3   d_SPGSCI                                                                          6135 non-null   float64
 4   m_Total refunds from the Income Tax Department                                    338 non-null    float64
 5   m_self employed returns                                                           338 non-nul

In [91]:
df_X = df_X.asfreq("D")

In [92]:
print(df_X.index.freq)

<Day>


In [93]:
df_X.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 11171 entries, 1995-01-01 to 2025-08-01
Freq: D
Data columns (total 47 columns):
 #   Column                                                                            Non-Null Count  Dtype  
---  ------                                                                            --------------  -----  
 0   d_TA125                                                                           4879 non-null   float64
 1   d_SP500                                                                           6135 non-null   float64
 2   d_USD_ILS                                                                         5969 non-null   float64
 3   d_SPGSCI                                                                          6135 non-null   float64
 4   m_Total refunds from the Income Tax Department                                    338 non-null    float64
 5   m_self employed returns                                                           33

## Publication Shifts

CPI is 15 days later  
Consumer Trust Index is 12 days later  
Unemployment rate is 16 days later  
Real Credit Card Usage is 3 days later


In [94]:
df_X.iloc[:, [17]] = df_X.iloc[:, [17]].shift(15)
df_X.iloc[:, [19]] = df_X.iloc[:, [19]].shift(12)
df_X.iloc[:, [12]] = df_X.iloc[:, [12]].shift(16)
df_X.iloc[:, [21]] = df_X.iloc[:, [21]].shift(3)

real salary delay is 50 days then drop it

In [82]:
df_X = df_X.drop(df.columns[26], axis=1)

drop the first 2 rows, and anything beyond 16.1.25

In [95]:
df_X = df_X.iloc[2:]
df_X = df_X[:'2025-01-16']

In [96]:
df_X

,d_TA125,d_SP500,d_USD_ILS,d_SPGSCI,m_Total refunds from the Income Tax Department,m_self employed returns,m_Cancellations Deductions,m_Capital Gas Tax Refunds,m_Cancellation companies,m_Purchase returns,...,m_Goods and services,m_Independents advances,m_Self-employed tax differences,m_Non-profit institution tax,m_tax differential companies,m_Companies advances,m_Income tax for self-employed dividuals and companies (advances and deductions),m_Deduction from salary,m_Deductions and the capital market,m_Total Income Tax Division Net
Date,,,,,,,,,,,,,,,,,,,,,
1995-01-03,0.001351,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,-0.000470,0.003479,0.001145,-0.004544,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,-0.011458,-0.000803,-0.001499,0.002863,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-06,NaN,0.000738,-0.000839,-0.007992,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-13,-0.000638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-14,0.014266,0.001146,-0.010987,-0.007369,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
df_X.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/df_X.pkl')

In [98]:
df_X_train = df_X[:'2021-12-31']
df_X_test = df_X['2022-01-01':]

In [99]:
df_X_train.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/df_X_train.pkl')

In [100]:
df_X_test.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/df_X_test.pkl')